# Emotion Model Training (Google Colab)

This notebook trains a small CNN on the FER-2013 dataset using TensorFlow Datasets.
After training, it exports `emotion_model.keras` for use in the Streamlit app.

**Steps**
1. Run all cells.
2. Download the exported model file.
3. Copy it into the project folder: `ml/emotion_model.keras`.


In [ ]:
!pip -q install tensorflow-datasets

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

IMG_SIZE = 48
BATCH_SIZE = 64

def preprocess(image, label):
    image = tf.image.rgb_to_grayscale(image)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds, test_ds = tfds.load(
    "fer2013",
    split=["train", "test"],
    as_supervised=True
)

train_ds = train_ds.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(7, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(train_ds, validation_data=test_ds, epochs=10)
model.save("emotion_model.keras")
print("Saved emotion_model.keras")


In [ ]:
from google.colab import files
files.download("emotion_model.keras")
